# 10 - Human Approval Packet For Conditional Export

Metatate decisions are operational. A `CONDITIONAL` result should turn into an approval packet with the source, destination, required controls, and evidence needed by the reviewer.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Request an external transfer decision


In [ ]:
transfer = client.authorize_use(
    "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS",
    operation="export",
    intended_use="external_sharing",
    actor_role="DATA_ENGINEER",
    columns=["CUSTOMER_ID", "CUSTOMER_NAME", "EMAIL", "ACCOUNT_STATUS"],
    destination={"system": "SALESFORCE", "jurisdiction": "US"},
    consumer_jurisdiction="EU",
)
print(json.dumps(transfer["data"], indent=2))


## Convert the decision into an approval packet


In [ ]:
def compact(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

data = transfer["data"]
decision_id = data.get("decision_id")
explanation = client.explain_why(decision_id=decision_id) if decision_id else {"data": {}}

approval_packet = {
    "title": "Approve controlled customer export to Salesforce",
    "decision": decision_label(transfer),
    "decision_id": decision_id,
    "source": data.get("table_name"),
    "destination": data.get("destination", {"system": "SALESFORCE", "jurisdiction": "US"}),
    "consumer_jurisdiction": data.get("consumer_jurisdiction", "EU"),
    "required_controls": compact(data.get("conditions")),
    "obligations": compact(data.get("obligations")),
    "rationale": rationale_text(transfer),
    "reviewer_note": agent_action_text(transfer),
    "explanation_summary": explanation.get("data", {}).get("summary") or explanation.get("data", {}).get("rationale"),
}

print(json.dumps(approval_packet, indent=2))


## Reviewer-facing summary


In [ ]:
lines = [
    f"# {approval_packet['title']}",
    f"Decision: {approval_packet['decision']}",
    f"Decision ID: {approval_packet['decision_id']}",
    f"Source: {approval_packet['source']}",
    f"Destination: {approval_packet['destination']}",
    "",
    "Required controls:",
]
lines.extend(f"- {control}" for control in approval_packet["required_controls"])
lines.extend(["", "Obligations:"])
lines.extend(f"- {obligation}" for obligation in approval_packet["obligations"])
lines.extend(["", f"Rationale: {approval_packet['rationale']}"])
print("\n".join(lines))


This is the bridge from agent output to governance operations. Conditional decisions become reviewable work items instead of ambiguous chat messages.
